# 📊 Evaluate Semantic Image Retrieval

Computes **Precision@K, Recall@K, MRR, and mAP** on Flickr30k.

This notebook works on **Google Colab** — it clones your project, installs dependencies, and runs evaluation.

> ⚠️ Set runtime to **GPU** for faster embedding generation: Runtime → Change runtime type → GPU

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q torch torchvision
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q faiss-cpu numpy pillow

In [ ]:
# Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU — evaluation will be slow but will work.")
    print("   Go to: Runtime → Change runtime type → GPU")

## 2. Mount Drive & Upload Project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# === OPTION A: Project is on Google Drive ===
# Set this to where your project folder is on Drive
PROJECT_DIR = '/content/drive/MyDrive/imageproject'

# === OPTION B: Upload/clone project to Colab ===
# Uncomment the line below if you want to clone from git instead:
# !git clone https://github.com/YOUR_USERNAME/imageproject.git /content/imageproject
# PROJECT_DIR = '/content/imageproject'

# Add project to Python path
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

print(f"Project directory: {PROJECT_DIR}")
print(f"Files: {os.listdir(PROJECT_DIR)[:10]}...")

## 3. Configure Paths

In [ ]:
from pathlib import Path
from datetime import datetime

project_root = Path(PROJECT_DIR)

# === Dataset paths ===
DATASET_DIR = str(project_root / 'data' / 'flickr30_data')
IMAGES_DIR = project_root / 'data' / 'flickr30_data' / 'flickr30k_images'
CAPTIONS_FILE = project_root / 'data' / 'flickr30_data' / 'captions.txt'

# Verify
for label, path in [('Images', IMAGES_DIR), ('Captions', CAPTIONS_FILE)]:
    exists = path.exists()
    status = '✅' if exists else '❌ MISSING'
    print(f"{status} {label}: {path}")

# === Evaluation storage (separate from main app) ===
from core import config

eval_storage = project_root / 'evaluation' / 'storage'
eval_storage.mkdir(parents=True, exist_ok=True)

config.STORAGE_DIR = eval_storage
config.EMBEDDINGS_PATH = config.STORAGE_DIR / 'embeddings.npy'
config.METADATA_PATH = config.STORAGE_DIR / 'metadata.json'
config.FAISS_INDEX_PATH = config.STORAGE_DIR / 'faiss.index'
config.FEEDBACK_PATH = config.STORAGE_DIR / 'feedback.json'
config.MODEL_FINGERPRINT_PATH = config.STORAGE_DIR / 'model_fingerprint.json'

eval_logs = project_root / 'evaluation' / 'logs'
eval_logs.mkdir(parents=True, exist_ok=True)
config.LOGS_DIR = eval_logs
timestamp = datetime.now().strftime('%Y_%m_%d_%H_%M_%S')
config.LOG_FILE = config.LOGS_DIR / f'evaluation_{timestamp}.log'

print(f"\nEval storage: {eval_storage}")
print(f"Model: {config.MODEL_NAME}")

## 4. Load Model & Build Index

In [ ]:
from core.clip_model import CLIPModel
from core.indexer import ImageIndexer
from core.search import ImageSearcher

print("Loading CLIP model...")
clip_model = CLIPModel(device=None)
print(f"Model loaded on: {clip_model.device}")

# Build index if not already cached
if not config.FAISS_INDEX_PATH.exists():
    print("\nBuilding FAISS index (this may take 10-20 min for 31K images)...")
    indexer = ImageIndexer(clip_model)
    
    def progress(cur, tot):
        if cur % 200 == 0 or cur == tot:
            print(f"  Indexed {cur:,}/{tot:,} images...")
    
    successful, failed = indexer.index_directory(str(IMAGES_DIR), progress_callback=progress)
    print(f"\n✅ Indexing complete: {successful:,} indexed, {failed} failed.")
else:
    print("✅ Existing FAISS index found. Skipping indexing.")

searcher = ImageSearcher(clip_model)
print(f"Searcher ready. Index has {searcher.index.ntotal:,} vectors.")

## 5. Run Evaluation

In [ ]:
import csv

# Parse captions — 1 caption per image for query-ground-truth pairs
query_image_map = []
seen_images = set()

with open(CAPTIONS_FILE, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    try:
        next(reader)  # skip header
    except StopIteration:
        pass
    
    for row in reader:
        if len(row) >= 2:
            img_name = row[0].strip()
            caption = row[1].strip()
            img_path = IMAGES_DIR / img_name
            
            if img_path.exists() and img_name not in seen_images:
                query_image_map.append((caption, str(img_path)))
                seen_images.add(img_name)

total_queries = len(query_image_map)
print(f"Loaded {total_queries:,} query-image pairs")

# Compute metrics
K_VALUES = [1, 5, 10]
results = {
    'P@1': 0.0, 'P@5': 0.0, 'P@10': 0.0,
    'R@1': 0.0, 'R@5': 0.0, 'R@10': 0.0,
    'MRR': 0.0, 'mAP': 0.0
}

max_k = max(K_VALUES)
print(f"Running {total_queries:,} search queries (top-{max_k})...\n")

for i, (query, ground_truth_path) in enumerate(query_image_map):
    search_results = searcher.search(query, top_k=max_k)
    retrieved_names = [Path(res[0]).name for res in search_results]
    gt_name = Path(ground_truth_path).name
    
    found_at_rank = -1
    for rank, name in enumerate(retrieved_names, 1):
        if name == gt_name:
            found_at_rank = rank
            break
    
    reciprocal_rank = (1.0 / found_at_rank) if found_at_rank != -1 else 0.0
    results['MRR'] += reciprocal_rank
    results['mAP'] += reciprocal_rank
    
    for k in K_VALUES:
        if found_at_rank != -1 and found_at_rank <= k:
            results[f'R@{k}'] += 1.0
            results[f'P@{k}'] += 1.0 / k
    
    if (i + 1) % 100 == 0 or (i + 1) == total_queries:
        print(f"  Processed {i+1:,}/{total_queries:,} queries...")

# Average
for k in results:
    results[k] /= total_queries

print("\nDone!")

## 6. Results

In [ ]:
print('\n' + '='*50)
print(' ' * 12 + 'EVALUATION RESULTS')
print('='*50)
print(f"Model        : {config.MODEL_NAME}")
print(f"Dataset      : {DATASET_DIR}")
print(f"Total Queries: {total_queries:,}")
print('-' * 50)
for k in K_VALUES:
    print(f"Precision@{k:2d} : {results[f'P@{k}']:.4f}")
print('-' * 50)
for k in K_VALUES:
    print(f"Recall@{k:2d}    : {results[f'R@{k}']:.4f}")
print('-' * 50)
print(f"MRR          : {results['MRR']:.4f}")
print(f"mAP          : {results['mAP']:.4f}")
print('='*50)

# Save results
results_dir = project_root / 'evaluation' / 'results'
results_dir.mkdir(exist_ok=True)
existing = list(results_dir.glob('experiment_*_results.csv'))
exp_num = len(existing) + 1
exp_file = results_dir / f'experiment_{exp_num}_results.csv'

ts = datetime.now()
with open(exp_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['date', 'time', 'model', 'num_images', 'num_queries',
                     'P@1', 'P@5', 'P@10', 'R@1', 'R@5', 'R@10', 'MRR', 'mAP'])
    writer.writerow([
        ts.strftime('%Y-%m-%d'), ts.strftime('%H:%M:%S'), config.MODEL_NAME,
        len(os.listdir(IMAGES_DIR)), total_queries,
        results['P@1'], results['P@5'], results['P@10'],
        results['R@1'], results['R@5'], results['R@10'],
        results['MRR'], results['mAP']
    ])

print(f"\nResults saved to: {exp_file}")